# OpenMP with Flang REPL

These are ordinary interactive Fortran cells. The hosted kernel starts Flang with `-fopenmp`, preloads `libomp`, and captures only the execution phase so threaded compilation and output forwarding do not interfere with each other.

In [1]:
integer :: thread_count
thread_count = 0
!$omp parallel
!$omp atomic update
thread_count = thread_count + 1
!$omp end atomic
!$omp end parallel
print *, 'OpenMP threads:', thread_count

 OpenMP threads: 10


In [2]:
subroutine reduction_demo()
  integer(kind=8) :: total, value
  total = 0
  !$omp parallel do reduction(+:total)
  do value = 1, 1000000
    total = total + value
  end do
  !$omp end parallel do
  print *, 'reduction:', total
end subroutine
call reduction_demo()

 reduction: 500000500000


In [3]:
subroutine sections_demo()
  integer :: first, second
  first = 0; second = 0
  !$omp parallel sections
  !$omp section
  first = 20
  !$omp section
  second = 22
  !$omp end parallel sections
  print *, 'sections:', first + second
end subroutine
call sections_demo()

 sections: 42


In [4]:
subroutine tasks_demo()
  integer :: values(4), task_index
  values = 0
  !$omp parallel
  !$omp single
  do task_index = 1, 4
    !$omp task firstprivate(task_index) shared(values)
    values(task_index) = task_index * task_index
    !$omp end task
  end do
  !$omp taskwait
  !$omp end single
  !$omp end parallel
  print *, 'tasks:', sum(values)
end subroutine
call tasks_demo()

 tasks: 30


The `%%mlir` magic lets us inspect the lowered OpenMP program without executing it.

In [5]:
%%mlir
subroutine openmp_sum(result)
  integer(kind=8), intent(out) :: result
  integer(kind=8) :: i
  result = 0
  !$omp parallel do reduction(+:result)
  do i = 1, 1000
    result = result + i
  end do
  !$omp end parallel do
end subroutine

#tbaa_root = #llvm.tbaa_root<id = "Flang function root _QMflang_repl_state_6Popenmp_sum">
#tbaa_type_desc = #llvm.tbaa_type_desc<id = "any access", members = {<#tbaa_root, 0>}>
#tbaa_type_desc1 = #llvm.tbaa_type_desc<id = "any data access", members = {<#tbaa_type_desc, 0>}>
#tbaa_tag = #llvm.tbaa_tag<base_type = #tbaa_type_desc1, access_type = #tbaa_type_desc1, offset = 0>
#tbaa_type_desc2 = #llvm.tbaa_type_desc<id = "dummy arg data", members = {<#tbaa_type_desc1, 0>}>
#tbaa_type_desc3 = #llvm.tbaa_type_desc<id = "dummy arg data/_QMflang_repl_state_6Fopenmp_sumEresult", members = {<#tbaa_type_desc2, 0>}>
#tbaa_tag1 = #llvm.tbaa_tag<base_type = #tbaa_type_desc3, access_type = #tbaa_type_desc3, offset = 0>
module @flang_repl_module_6 attributes {dlti.dl_spec = #dlti.dl_spec<!llvm.ptr<270> = dense<32> : vector<4xi64>, !llvm.ptr<271> = dense<32> : vector<4xi64>, !llvm.ptr<272> = dense<64> : vector<4xi64>, i64 = dense<64> : vector<2xi64>, i128 = dense<128> : vector<2xi64>, !llvm.ptr = dense